In [1]:
# ============================================================
# Standard NVAR = NG-RC (Aligned for Fair Comparison)
# ============================================================
import warnings
import numpy as np
import optuna

warnings.filterwarnings("ignore", category=FutureWarning)

# Precision Settings
NP_DTYPE = np.float32

# ============================================================
# 1. Reproducibility & Data Loading
# ============================================================
base_seed = 2025
np.random.seed(base_seed)

# Load Clean Reference
data_clean = np.load("mg_noise_0.npy", allow_pickle=True).item()
mg = data_clean["data"]
if mg.shape[0] == 1:  # Force (T, d) layout standard to match Adaptive NVAR code
    mg = mg.T
mg = mg.astype(NP_DTYPE)

# Load Noisy Dataset (20% global measurement noise added)
data_noisy = np.load("mg_noise_20.npy", allow_pickle=True).item()
X_noisy_dataset = data_noisy["data"]
if X_noisy_dataset.shape[0] == 1:
    X_noisy_dataset = X_noisy_dataset.T
X_noisy_dataset = X_noisy_dataset.astype(NP_DTYPE)

# Match exact splitting boundary points from PyTorch script
warmup_len, train_len, val_len, test_len = 500, 7500, 1000, 1000

X_train = X_noisy_dataset[warmup_len : warmup_len + train_len]
X_val   = X_noisy_dataset[warmup_len + train_len : warmup_len + train_len + val_len]
X_test  = X_noisy_dataset[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len]

X_val_true  = mg[warmup_len + train_len : warmup_len + train_len + val_len]
X_test_true = mg[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len]

# Structural Parameters
d = 1
horizons = [25, 50, 75, 100]

# ============================================================
# 2. Linear and Nonlinear Feature Extraction Functions
# ============================================================
def build_nvar_features(X_tensor, k):
    """Builds polynomial features from history layout matching layout standard"""
    T = X_tensor.shape[0]
    # Linear Delay Vectors
    H_lin = []
    for t in range(k - 1, T - 1):
        delays = [X_tensor[t - delay] for delay in range(k)]
        H_lin.append(np.concatenate(delays, axis=0))
    H_lin = np.stack(H_lin)  # Shape: (T_reduced, d * k)

    T_reduced, dlin = H_lin.shape
    dnonlin = dlin * (dlin + 1) // 2
    dtot = 1 + dlin + dnonlin
    
    # Pre-allocate total feature matrix for speed
    out = np.ones((T_reduced, dtot), dtype=NP_DTYPE)
    out[:, 1:1 + dlin] = H_lin
    
    # Construct exact unique unique quadratic polynomial library
    cnt = 0
    for r in range(dlin):
        for c in range(r, dlin):
            out[:, 1 + dlin + cnt] = H_lin[:, r] * H_lin[:, c]
            cnt += 1
            
    return out

# ============================================================
# 3. Training Engine (State-to-State Configured via Ridge Regression)
# ============================================================
def train_standard_nvar(X_input, k, ridge_param):
    # Feature construction matrix
    O_train = build_nvar_features(X_input, k)
    
    # FIXED: Map to full absolute next states instead of noisy delta
    Y_train = X_input[k:] 
    
    C = O_train.T @ O_train
    regularizer = ridge_param * np.eye(C.shape[0], dtype=NP_DTYPE)
    
    try:
        W_out = np.linalg.solve(C + regularizer, O_train.T @ Y_train)
    except np.linalg.LinAlgError:
        W_out = np.linalg.pinv(C + regularizer) @ O_train.T @ Y_train
        
    return W_out

# ============================================================
# 4. Core Evaluation Pipeline Function (Overlapping Sliding Windows)
# ============================================================
def evaluate_nvar_model(W_out, k, X_true_target, X_history, horizon_max, stride, h_list):
    horizon_rmses = {h: [] for h in h_list}
    total_len = len(X_true_target)
    
    dlin = k * d
    dtot = 1 + dlin + (dlin * (dlin + 1) // 2)

    # Slide through validation set with a stride of 10 steps for dense comparison
    for start_idx in range(k, total_len - horizon_max + 1, stride):
        y_true_window = X_true_target[start_idx : start_idx + horizon_max]
        X_init = X_history[start_idx - k : start_idx]
        
        x_t = [x.copy() for x in X_init]
        
        predictions = []
        for _ in range(horizon_max):
            z = np.concatenate([x_t[-1 - delay] for delay in range(k)], axis=0)
            
            # Inline construction for single test step
            out_test = np.ones(dtot, dtype=NP_DTYPE)
            out_test[1:1 + dlin] = z
            
            cnt = 0
            for r in range(dlin):
                for c in range(r, dlin):
                    out_test[1 + dlin + cnt] = z[r] * z[c]
                    cnt += 1
            
            # Predict direct absolute coordinate state
            x_next = out_test @ W_out
            predictions.append(x_next)
            
            # Append predictive coordinate back to sequence loop
            x_t = x_t[1:] + [x_next]
            
        predictions_np = np.stack(predictions)
        
        for h in h_list:
            rmse = np.sqrt(np.mean((predictions_np[:h] - y_true_window[:h]) ** 2))
            horizon_rmses[h].append(rmse)
            
    return {h: np.mean(horizon_rmses[h]) for h in h_list}

def objective(trial):
    k_suggest = trial.suggest_categorical("k", [2, 5, 10, 20, 30, 40, 50])
    ridge_suggest = trial.suggest_categorical("ridge_param", [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4])
    
    W_out = train_standard_nvar(X_train, k_suggest, ridge_suggest)
    
    rmse_by_horizon = evaluate_nvar_model(
        W_out=W_out, k=k_suggest,
        X_true_target=X_val_true, X_history=X_val,
        horizon_max=100, stride=10, h_list=horizons
    )
    return rmse_by_horizon[100]

# ============================================================
# 5. Automated Execution Workflow
# ============================================================
if __name__ == "__main__":
    print("Starting Automated Optuna Standard NVAR Parameter Search...\n")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=base_seed)
    )
    study.optimize(objective, n_trials=20)

    best_trial = study.best_trial
    best_params = best_trial.params

    print("\n" + "=" * 60)
    print("BEST STANDARD NVAR CONFIGURATION RETRIEVED")
    print("=" * 60)
    print(f"k          : {best_params['k']}")
    print(f"ridge      : {best_params['ridge_param']:.6e}")
    print(f"Validation Target Average RMSE@100: {best_trial.value:.6f}")

    print("\n=== Launching Final Test Benchmark Using Best Values ===")
    k_best = best_params['k']
    ridge_best = best_params['ridge_param']
    
    # Evaluate over the test block using the exact sliding stride setup
    rmse_by_horizon = evaluate_nvar_model(
        W_out=train_standard_nvar(X_train, k_best, ridge_best),
        k=k_best,
        X_true_target=X_test_true, X_history=X_test,
        horizon_max=100, stride=10, h_list=horizons
    )

    print("\n" + "=" * 60)
    print("FINAL STANDARD NVAR TEST EXTRAPOLATION SUMMARY")
    print("=" * 60)
    print(f"Parameters utilized: k={k_best}, ridge={ridge_best:.6e}")
    print("-" * 60)
    for h in horizons:
        print(f"Horizon {h:3d} steps: {rmse_by_horizon[h]:.6f}")

/home/eric/.conda/envs/nvar/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-29 18:38:48,652] A new study created in memory with name: no-name-44ab8775-8610-4c3a-a1ac-9cdcff8998bb


Starting Automated Optuna Standard NVAR Parameter Search...



[I 2026-06-29 18:38:48,952] Trial 0 finished with value: 0.21447408199310303 and parameters: {'k': 10, 'ridge_param': 1e-06}. Best is trial 0 with value: 0.21447408199310303.
[I 2026-06-29 18:38:49,252] Trial 1 finished with value: 0.2149169147014618 and parameters: {'k': 10, 'ridge_param': 0.01}. Best is trial 0 with value: 0.21447408199310303.
/tmp/ipykernel_119160/4217430156.py:125: RuntimeWarning: overflow encountered in scalar multiply
  out_test[1 + dlin + cnt] = z[r] * z[c]
/tmp/ipykernel_119160/4217430156.py:138: RuntimeWarning: overflow encountered in square
  rmse = np.sqrt(np.mean((predictions_np[:h] - y_true_window[:h]) ** 2))
[I 2026-06-29 18:38:49,555] Trial 2 finished with value: inf and parameters: {'k': 10, 'ridge_param': 10000.0}. Best is trial 0 with value: 0.21447408199310303.
/tmp/ipykernel_119160/4217430156.py:129: RuntimeWarning: overflow encountered in matmul
  x_next = out_test @ W_out
/tmp/ipykernel_119160/4217430156.py:129: RuntimeWarning: invalid value encou


BEST STANDARD NVAR CONFIGURATION RETRIEVED
k          : 50
ridge      : 1.000000e-01
Validation Target Average RMSE@100: 0.058716

=== Launching Final Test Benchmark Using Best Values ===

FINAL STANDARD NVAR TEST EXTRAPOLATION SUMMARY
Parameters utilized: k=50, ridge=1.000000e-01
------------------------------------------------------------
Horizon  25 steps: 0.033989
Horizon  50 steps: 0.042718
Horizon  75 steps: 0.053734
Horizon 100 steps: 0.066868
